In [ ]:
%pip install -q -U "transformers>=5.12" accelerate bitsandbytes huggingface_hub "pillow<12"

## Local Inference on GPU 
Model page: https://huggingface.co/orcarouter/Qwen3.8-27B-Uncensored

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/orcarouter/Qwen3.8-27B-Uncensored)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

The model you are trying to use is gated. Please make sure you have access to it by visiting the model page.To run inference, either set HF_TOKEN in your environment variables/ Secrets or run the following cell to login. 🤗

In [ ]:
# Authenticate Hugging Face without putting the token in notebook source.
import os
from huggingface_hub import login

def load_secret(*names):
    for name in names:
        value = os.environ.get(name)
        if value:
            return value
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in names:
            try:
                value = client.get_secret(name)
            except Exception:
                continue
            if value:
                return value
    except Exception:
        pass
    return None

token = load_secret("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")
if not token:
    raise RuntimeError(
        "Missing HF_TOKEN. In this Kaggle notebook open Add-ons → Secrets and add HF_TOKEN "
        "(a Hugging Face token that can read orcarouter/Qwen3.8-27B-Uncensored)."
    )
login(token=token)
print("Hugging Face login OK")


In [ ]:
# Load one 4-bit model across the available T4 GPUs.
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "orcarouter/Qwen3.8-27B-Uncensored"
assert torch.cuda.is_available(), "Select GPU T4 x2 in Session options."
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    max_memory={i: "14GiB" for i in range(torch.cuda.device_count())},
    dtype=torch.float16,
    attn_implementation="sdpa",
)
model.eval()
print("Model loaded:", model.hf_device_map)

In [ ]:
# Run this cell again to ask another question; the model stays loaded.
import gc
import requests
from PIL import Image
from io import BytesIO


gc.collect()
torch.cuda.empty_cache()
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
response = requests.get(image_url, timeout=60)
response.raise_for_status()
image = Image.open(BytesIO(response.content)).convert("RGB")
image.thumbnail((384, 384))  # Keep vision attention within T4 memory limits.
messages = [{"role": "user", "content": [
    {"type": "image", "image": image},
    {"type": "text", "text": "What animal is on the candy? Answer in one short sentence."},
]}]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)
for key, value in inputs.items():
    if torch.is_tensor(value) and value.is_floating_point():
        inputs[key] = value.to(torch.float16)
print("Image size:", image.size, "Input tokens:", inputs["input_ids"].shape[-1])
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

In [ ]:
%pip install -q fastapi uvicorn pyngrok
import fastapi, uvicorn, pyngrok
print("API dependencies ready")

In [ ]:
# OpenAI-compatible text API: Chat Completions + Responses. Reuses the loaded 4-bit model.
import json, time, uuid, threading, queue, gc
from collections import OrderedDict
from typing import Any, Literal
from fastapi import FastAPI, HTTPException
from fastapi.responses import Response, StreamingResponse
from pydantic import BaseModel, Field
from transformers import TextIteratorStreamer, StoppingCriteria, StoppingCriteriaList
import uvicorn

API_MODEL = MODEL_ID
api_tokenizer = processor.tokenizer
api_lock = threading.Lock()
response_store = OrderedDict()
store_lock = threading.Lock()
RESPONSE_STORE_LIMIT = 32
app = FastAPI(title="Kaggle Qwen 27B API")

class ChatRequest(BaseModel):
    model: str = API_MODEL
    messages: list[dict[str, Any]] = Field(min_length=1, max_length=100)
    max_tokens: int = Field(default=256, ge=1, le=512)
    temperature: float = Field(default=0.6, ge=0, le=2)
    top_p: float = Field(default=0.95, gt=0, le=1)
    stream: bool = False
    stop: str | list[str] | None = None
    tools: list[dict[str, Any]] | None = None
    tool_choice: Literal["auto", "none"] = "auto"

class ResponseCreateRequest(BaseModel):
    model: str = API_MODEL
    input: str | list[Any] | None = None
    instructions: str | None = None
    max_output_tokens: int = Field(default=256, ge=1, le=512)
    temperature: float = Field(default=0.6, ge=0, le=2)
    top_p: float = Field(default=0.95, gt=0, le=1)
    stream: bool = False
    tools: list[dict[str, Any]] | None = None
    tool_choice: Any = "auto"
    previous_response_id: str | None = None
    store: bool = True
    metadata: dict[str, Any] | None = None
    background: bool | None = None
    conversation: Any = None
    text: Any = None

@app.get("/")
@app.get("/health")
def health():
    return {"status": "ok", "model": API_MODEL, "busy": api_lock.locked()}

@app.get("/v1/models")
def list_models():
    return {"object": "list", "data": [{"id": API_MODEL, "object": "model", "created": int(time.time()), "owned_by": "local"}]}

class CancelGeneration(StoppingCriteria):
    def __init__(self, event):
        self.event = event
    def __call__(self, input_ids, scores, **kwargs):
        return self.event.is_set()

def parse_calls(text):
    decoder = json.JSONDecoder()
    for index, char in enumerate(text):
        if char != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[index:])
            calls = value.get("tool_calls", [])
            if not calls:
                continue
            result = []
            for call in calls:
                function = call.get("function", call)
                arguments = function.get("arguments", {})
                result.append({"id": "call_" + uuid.uuid4().hex[:12], "type": "function", "function": {
                    "name": function["name"],
                    "arguments": arguments if isinstance(arguments, str) else json.dumps(arguments),
                }})
            return result
        except (ValueError, KeyError, AttributeError, TypeError):
            continue
    return []

def start_generation(messages, *, max_tokens, temperature, top_p, stops):
    if sum(len(m["content"]) for m in messages) > 20000:
        raise HTTPException(413, "Request is too long for this T4 demo.")
    encoded = api_tokenizer.apply_chat_template(messages, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt", enable_thinking=False)
    prompt_tokens = encoded["input_ids"].shape[-1]
    if prompt_tokens > 2048:
        raise HTTPException(413, "Maximum input length is 2048 tokens on this T4 demo.")
    if not api_lock.acquire(blocking=False):
        raise HTTPException(429, "Model is busy; retry after the current request finishes.")
    cancel = threading.Event()
    state = {}
    try:
        inputs = encoded.to(model.device)
        streamer = TextIteratorStreamer(api_tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=1)
        options = dict(max_new_tokens=max_tokens, do_sample=temperature > 0,
            streamer=streamer, stopping_criteria=StoppingCriteriaList([CancelGeneration(cancel)]))
        if temperature > 0:
            options.update(temperature=temperature, top_p=top_p)
        if stops:
            options.update(stop_strings=stops, tokenizer=api_tokenizer)
        def worker():
            try:
                with torch.inference_mode():
                    generated = model.generate(**inputs, **options)
                state["tokens"] = generated.shape[-1] - prompt_tokens
            except Exception as exc:
                state["error"] = type(exc).__name__
                streamer.end()
            finally:
                api_lock.release()
        thread = threading.Thread(target=worker, daemon=True)
        thread.start()
    except Exception:
        api_lock.release()
        raise
    return {"thread": thread, "streamer": streamer, "state": state, "cancel": cancel, "prompt_tokens": prompt_tokens}

def iter_pieces(job, stops):
    streamer, thread, state, cancel = job["streamer"], job["thread"], job["state"], job["cancel"]
    pending = ""
    hold = max(map(len, stops), default=0)
    try:
        while True:
            try:
                piece = next(streamer)
            except queue.Empty:
                if not thread.is_alive():
                    break
                continue
            except StopIteration:
                break
            pending += piece
            matches = [pending.index(s) for s in stops if s in pending]
            if matches:
                yield pending[:min(matches)]
                cancel.set()
                pending = ""
                break
            safe = len(pending) - hold
            if safe > 0:
                yield pending[:safe]
                pending = pending[safe:]
        if pending:
            yield pending
        thread.join()
        if "error" in state:
            raise RuntimeError(state["error"])
    finally:
        cancel.set()

def remember_response(response_obj, input_items, *, store):
    if not store:
        return
    with store_lock:
        response_store[response_obj["id"]] = {"response": response_obj, "input": input_items}
        while len(response_store) > RESPONSE_STORE_LIMIT:
            response_store.popitem(last=False)

def load_stored(response_id):
    with store_lock:
        return response_store.get(response_id)

def flatten_content(content):
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, str):
                parts.append(part)
                continue
            if not isinstance(part, dict):
                raise HTTPException(422, "Unsupported content part.")
            ptype = part.get("type")
            if ptype in {"input_image", "input_file", "output_image", "refusal"}:
                raise HTTPException(422, "This endpoint accepts text messages only.")
            if ptype in {"input_text", "output_text", "text"} or "text" in part:
                parts.append(part.get("text") or "")
            else:
                raise HTTPException(422, "Unsupported content part.")
        return "".join(parts)
    raise HTTPException(422, "Unsupported content.")

def item_to_messages(item):
    if not isinstance(item, dict):
        raise HTTPException(422, "Invalid input item.")
    itype = item.get("type") or ("message" if "role" in item else None)
    if itype in {None, "message"}:
        role = item.get("role") or "user"
        if role == "developer":
            role = "system"
        if role not in {"system", "user", "assistant", "tool"}:
            raise HTTPException(422, "Unsupported message role.")
        return [{"role": role, "content": flatten_content(item.get("content"))}]
    if itype == "function_call":
        arguments = item.get("arguments", "{}")
        if not isinstance(arguments, str):
            arguments = json.dumps(arguments)
        return [{"role": "assistant", "content": json.dumps({"tool_calls": [{
            "id": item.get("call_id") or item.get("id"),
            "type": "function",
            "function": {"name": item.get("name"), "arguments": arguments},
        }]})}]
    if itype == "function_call_output":
        output = item.get("output")
        if isinstance(output, list):
            output = flatten_content(output)
        elif output is None:
            output = ""
        elif not isinstance(output, str):
            output = json.dumps(output)
        return [{"role": "tool", "content": output}]
    raise HTTPException(422, f"Unsupported input item type: {itype}. This T4 demo supports messages, function_call, and function_call_output.")

def normalize_input_items(raw):
    if raw is None or raw == "":
        return []
    if isinstance(raw, str):
        raw = [{"type": "message", "role": "user", "content": raw}]
    if not isinstance(raw, list):
        raise HTTPException(422, "input must be a string or an array of items.")
    if len(raw) > 100:
        raise HTTPException(422, "Too many input items for this T4 demo.")
    items = []
    for item in raw:
        if isinstance(item, str):
            item = {"type": "message", "role": "user", "content": item}
        if not isinstance(item, dict):
            raise HTTPException(422, "Invalid input item.")
        item = dict(item)
        item.setdefault("type", "message" if "role" in item else None)
        if not item.get("id"):
            prefix = "msg" if item.get("type") in {None, "message"} else "item"
            item["id"] = prefix + "_" + uuid.uuid4().hex[:16]
        items.append(item)
    return items

def expand_previous(prev_id):
    chain, seen = [], set()
    while prev_id:
        if prev_id in seen:
            raise HTTPException(400, "previous_response_id cycle.")
        seen.add(prev_id)
        stored = load_stored(prev_id)
        if not stored:
            raise HTTPException(404, "previous_response_id was not found. This demo keeps the last 32 stored responses in memory.")
        chain.append(stored)
        prev_id = stored["response"].get("previous_response_id")
    items = []
    for stored in reversed(chain):
        items.extend(stored["input"])
        items.extend(stored["response"].get("output") or [])
    return items

def responses_tools(tools):
    if not tools:
        return []
    normalized = []
    for tool in tools:
        if not isinstance(tool, dict):
            raise HTTPException(422, "Invalid tool.")
        ttype = tool.get("type") or "function"
        if ttype != "function":
            raise HTTPException(422, "Only function tools are supported on this T4 demo.")
        if "function" in tool:
            normalized.append(tool)
            continue
        name = tool.get("name")
        if not name:
            raise HTTPException(422, "Function tools require a name.")
        normalized.append({"type": "function", "function": {
            "name": name,
            "description": tool.get("description") or "",
            "parameters": tool.get("parameters") or {"type": "object", "properties": {}},
        }})
    return normalized

def tool_choice_none(choice):
    return choice == "none" or (isinstance(choice, dict) and choice.get("type") == "none")

def text_output_item(msg_id, text, status="completed"):
    return {"id": msg_id, "type": "message", "status": status, "role": "assistant",
            "content": [{"type": "output_text", "text": text, "annotations": []}]}

def function_output_items(calls):
    items = []
    for call in calls:
        function = call["function"]
        items.append({
            "id": "fc_" + uuid.uuid4().hex[:16],
            "type": "function_call",
            "status": "completed",
            "call_id": call["id"],
            "name": function["name"],
            "arguments": function["arguments"] if isinstance(function["arguments"], str) else json.dumps(function["arguments"]),
        })
    return items

def usage_block(prompt_tokens, output_tokens):
    return {"input_tokens": prompt_tokens, "input_tokens_details": {"cached_tokens": 0},
            "output_tokens": output_tokens, "output_tokens_details": {"reasoning_tokens": 0},
            "total_tokens": prompt_tokens + output_tokens}

def build_response(*, response_id, created, status, output, usage, req, incomplete_details=None, error=None):
    text = req.text if isinstance(req.text, dict) else {"format": {"type": "text"}}
    return {
        "id": response_id, "object": "response", "created_at": created, "status": status,
        "error": error, "incomplete_details": incomplete_details, "instructions": req.instructions,
        "max_output_tokens": req.max_output_tokens, "model": API_MODEL, "output": output,
        "parallel_tool_calls": True, "previous_response_id": req.previous_response_id,
        "reasoning": None, "store": req.store, "temperature": req.temperature, "text": text,
        "tool_choice": req.tool_choice if req.tool_choice is not None else "auto",
        "tools": req.tools or [], "top_p": req.top_p, "truncation": "disabled",
        "usage": usage, "metadata": req.metadata or {},
    }

def sse(event_type, payload, seq):
    body = {**payload, "type": event_type, "sequence_number": seq}
    return f"event: {event_type}\ndata: {json.dumps(body)}\n\n"

@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    if req.model != API_MODEL:
        raise HTTPException(404, "Unknown model. See /v1/models.")
    stops = [req.stop] if isinstance(req.stop, str) else (req.stop or [])
    if len(stops) > 4 or any(not s or len(s) > 200 for s in stops):
        raise HTTPException(422, "Use up to four nonempty stop strings, each at most 200 characters.")
    messages = []
    for message in req.messages:
        if message.get("role") not in {"system", "user", "assistant", "tool"}:
            raise HTTPException(422, "Unsupported message role.")
        content = message.get("content") or ""
        if not isinstance(content, str):
            raise HTTPException(422, "This endpoint accepts text messages only.")
        if message.get("tool_calls"):
            content += json.dumps({"tool_calls": message["tool_calls"]})
        messages.append({"role": message["role"], "content": content})
    use_tools = bool(req.tools) and req.tool_choice != "none"
    if use_tools:
        instruction = ('Available tools: ' + json.dumps(req.tools) +
            '\nWhen a tool is needed, respond ONLY with JSON in this format: '
            '{"tool_calls":[{"name":"function_name","arguments":{"key":"value"}}]}. '
            'Otherwise answer normally. Tool results are provided in tool messages.')
        messages.insert(0, {"role": "system", "content": instruction})
    job = start_generation(messages, max_tokens=req.max_tokens, temperature=req.temperature,
        top_p=req.top_p, stops=stops)
    request_id, created = "chatcmpl-" + uuid.uuid4().hex[:16], int(time.time())
    base = {"id": request_id, "created": created, "model": API_MODEL}
    def finish():
        return "length" if job["state"].get("tokens", 0) >= req.max_tokens else "stop"
    if not req.stream:
        try:
            text = "".join(iter_pieces(job, stops))
        except RuntimeError as exc:
            raise HTTPException(500, "Generation failed: " + str(exc))
        calls = parse_calls(text) if use_tools else []
        message = {"role": "assistant", "content": None if calls else text}
        if calls:
            message["tool_calls"] = calls
        completion = job["state"].get("tokens", 0)
        return {**base, "object": "chat.completion", "choices": [{"index": 0, "message": message,
            "finish_reason": "tool_calls" if calls else finish()}],
            "usage": {"prompt_tokens": job["prompt_tokens"], "completion_tokens": completion,
                      "total_tokens": job["prompt_tokens"] + completion}}
    def event(delta, reason=None):
        return "data: " + json.dumps({**base, "object": "chat.completion.chunk", "choices": [
            {"index": 0, "delta": delta, "finish_reason": reason}]}) + "\n\n"
    def events():
        try:
            yield event({"role": "assistant"})
            if use_tools:
                text = "".join(iter_pieces(job, stops))
                calls = parse_calls(text)
                if calls:
                    yield event({"tool_calls": [{"index": i, **call} for i, call in enumerate(calls)]})
                else:
                    yield event({"content": text})
                yield event({}, "tool_calls" if calls else finish())
            else:
                for piece in iter_pieces(job, stops):
                    if piece:
                        yield event({"content": piece})
                yield event({}, finish())
            yield "data: [DONE]\n\n"
        except Exception as exc:
            yield "data: " + json.dumps({"error": {"message": "Generation failed: " + type(exc).__name__}}) + "\n\n"
        finally:
            job["cancel"].set()
    return StreamingResponse(events(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

def responses_messages(req: ResponseCreateRequest):
    if req.background:
        raise HTTPException(400, "background responses are not supported on this T4 demo.")
    if req.conversation is not None:
        raise HTTPException(400, "conversation is not supported; use previous_response_id.")
    input_items = normalize_input_items(req.input)
    if not input_items and not req.previous_response_id:
        raise HTTPException(422, "Provide input or previous_response_id.")
    history = expand_previous(req.previous_response_id) if req.previous_response_id else []
    messages = []
    if req.instructions:
        messages.append({"role": "system", "content": req.instructions})
    for item in history + input_items:
        messages.extend(item_to_messages(item))
    tools = responses_tools(req.tools)
    use_tools = bool(tools) and not tool_choice_none(req.tool_choice)
    if use_tools:
        instruction = ('Available tools: ' + json.dumps(tools) +
            '\nWhen a tool is needed, respond ONLY with JSON in this format: '
            '{"tool_calls":[{"name":"function_name","arguments":{"key":"value"}}]}. '
            'Otherwise answer normally. Tool results are provided in function_call_output items.')
        if isinstance(req.tool_choice, dict) and req.tool_choice.get("type") == "function" and req.tool_choice.get("name"):
            instruction += f' Prefer calling {req.tool_choice["name"]} when a tool is needed.'
        messages.insert(0, {"role": "system", "content": instruction})
    return input_items, use_tools, messages

def responses_output(text, use_tools, max_output_tokens, generated_tokens):
    calls = parse_calls(text) if use_tools else []
    truncated = generated_tokens >= max_output_tokens
    if calls:
        return function_output_items(calls), "completed", None
    status = "incomplete" if truncated else "completed"
    details = {"reason": "max_output_tokens"} if truncated else None
    return [text_output_item("msg_" + uuid.uuid4().hex[:16], text)], status, details

@app.post("/v1/responses")
def create_response(req: ResponseCreateRequest):
    if req.model != API_MODEL:
        raise HTTPException(404, "Unknown model. See /v1/models.")
    input_items, use_tools, messages = responses_messages(req)
    job = start_generation(messages, max_tokens=req.max_output_tokens, temperature=req.temperature,
        top_p=req.top_p, stops=[])
    response_id, created, msg_id = "resp_" + uuid.uuid4().hex[:24], int(time.time()), "msg_" + uuid.uuid4().hex[:16]
    def completed_from(text):
        output, status, details = responses_output(text, use_tools, req.max_output_tokens, job["state"].get("tokens", 0))
        if output and output[0].get("type") == "message":
            output[0]["id"] = msg_id
        usage = usage_block(job["prompt_tokens"], job["state"].get("tokens", 0))
        return build_response(response_id=response_id, created=created, status=status, output=output,
            usage=usage, req=req, incomplete_details=details)
    if not req.stream:
        try:
            text = "".join(iter_pieces(job, []))
        except RuntimeError as exc:
            raise HTTPException(500, "Generation failed: " + str(exc))
        response_obj = completed_from(text)
        remember_response(response_obj, input_items, store=req.store)
        return response_obj
    def events():
        seq = 0
        empty = build_response(response_id=response_id, created=created, status="in_progress",
            output=[], usage=None, req=req)
        try:
            yield sse("response.created", {"response": empty}, seq); seq += 1
            yield sse("response.in_progress", {"response": empty}, seq); seq += 1
            if use_tools:
                text = "".join(iter_pieces(job, []))
                response_obj = completed_from(text)
                for index, item in enumerate(response_obj["output"]):
                    added = dict(item)
                    if item["type"] == "function_call":
                        added = {**item, "arguments": "", "status": "in_progress"}
                    elif item["type"] == "message":
                        added = {**item, "status": "in_progress", "content": []}
                    yield sse("response.output_item.added", {"output_index": index, "item": added}, seq); seq += 1
                    if item["type"] == "message":
                        part = item["content"][0]
                        yield sse("response.content_part.added", {"item_id": item["id"], "output_index": index,
                            "content_index": 0, "part": {"type": "output_text", "text": "", "annotations": []}}, seq); seq += 1
                        if part["text"]:
                            yield sse("response.output_text.delta", {"item_id": item["id"], "output_index": index,
                                "content_index": 0, "delta": part["text"]}, seq); seq += 1
                        yield sse("response.output_text.done", {"item_id": item["id"], "output_index": index,
                            "content_index": 0, "text": part["text"]}, seq); seq += 1
                        yield sse("response.content_part.done", {"item_id": item["id"], "output_index": index,
                            "content_index": 0, "part": part}, seq); seq += 1
                    else:
                        yield sse("response.function_call_arguments.delta", {"item_id": item["id"],
                            "output_index": index, "delta": item["arguments"]}, seq); seq += 1
                        yield sse("response.function_call_arguments.done", {"item_id": item["id"],
                            "output_index": index, "arguments": item["arguments"]}, seq); seq += 1
                    yield sse("response.output_item.done", {"output_index": index, "item": item}, seq); seq += 1
            else:
                yield sse("response.output_item.added", {"output_index": 0, "item": {
                    "id": msg_id, "type": "message", "status": "in_progress", "role": "assistant", "content": []}}, seq); seq += 1
                yield sse("response.content_part.added", {"item_id": msg_id, "output_index": 0, "content_index": 0,
                    "part": {"type": "output_text", "text": "", "annotations": []}}, seq); seq += 1
                text = ""
                for piece in iter_pieces(job, []):
                    if not piece:
                        continue
                    text += piece
                    yield sse("response.output_text.delta", {"item_id": msg_id, "output_index": 0,
                        "content_index": 0, "delta": piece}, seq); seq += 1
                response_obj = completed_from(text)
                item = response_obj["output"][0] if response_obj["output"] else text_output_item(msg_id, text)
                part = item["content"][0]
                yield sse("response.output_text.done", {"item_id": msg_id, "output_index": 0,
                    "content_index": 0, "text": part["text"]}, seq); seq += 1
                yield sse("response.content_part.done", {"item_id": msg_id, "output_index": 0,
                    "content_index": 0, "part": part}, seq); seq += 1
                yield sse("response.output_item.done", {"output_index": 0, "item": item}, seq); seq += 1
            remember_response(response_obj, input_items, store=req.store)
            yield sse("response.completed", {"response": response_obj}, seq)
            yield "data: [DONE]\n\n"
        except Exception as exc:
            failed = build_response(response_id=response_id, created=created, status="failed", output=[],
                usage=None, req=req, error={"code": "server_error", "message": "Generation failed: " + type(exc).__name__})
            yield sse("response.failed", {"response": failed}, seq)
            yield "data: [DONE]\n\n"
        finally:
            job["cancel"].set()
    return StreamingResponse(events(), media_type="text/event-stream", headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

@app.get("/v1/responses/{response_id}")
def retrieve_response(response_id: str):
    stored = load_stored(response_id)
    if not stored:
        raise HTTPException(404, "Response not found.")
    return stored["response"]

@app.delete("/v1/responses/{response_id}")
def delete_response(response_id: str):
    with store_lock:
        if response_id not in response_store:
            raise HTTPException(404, "Response not found.")
        del response_store[response_id]
    return Response(status_code=204)

@app.get("/v1/responses/{response_id}/input_items")
def list_input_items(response_id: str):
    stored = load_stored(response_id)
    if not stored:
        raise HTTPException(404, "Response not found.")
    data = stored["input"]
    return {"object": "list", "data": data,
            "first_id": data[0]["id"] if data else None,
            "last_id": data[-1]["id"] if data else None, "has_more": False}

# Bind only the API to loopback; ngrok will forward to this port.
if "api_server" in globals():
    api_server.should_exit = True
    api_thread.join(timeout=10)
api_server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning"))
api_thread = threading.Thread(target=api_server.run, daemon=True)
api_thread.start()
for _ in range(100):
    if api_server.started:
        break
    time.sleep(0.1)
assert api_server.started, "API server failed to start"
print("API ready at http://127.0.0.1:8000; model:", API_MODEL)
print("Chat Completions: POST /v1/chat/completions")
print("Responses: POST /v1/responses")


In [ ]:
# Verify chat, Responses API, real SSE streaming, stop handling, and tool-call formatting.
import requests, json
base_url = "http://127.0.0.1:8000"
assert requests.get(base_url + "/health", timeout=10).json()["status"] == "ok"
assert requests.get(base_url + "/v1/models", timeout=10).json()["data"][0]["id"] == API_MODEL
payload = {"model": API_MODEL, "messages": [{"role": "user", "content": "What is 2 + 2? Reply with only the number."}], "max_tokens": 32, "temperature": 0}
r = requests.post(base_url + "/v1/chat/completions", json=payload, timeout=180)
r.raise_for_status()
print("CHAT:", r.json()["choices"][0]["message"])
with requests.post(base_url + "/v1/chat/completions", json={**payload, "stream": True}, stream=True, timeout=180) as r:
    r.raise_for_status()
    chunks, done = [], False
    for line in r.iter_lines():
        if line == b"data: [DONE]":
            done = True
        elif line.startswith(b"data: "):
            chunk = json.loads(line[6:])
            assert "error" not in chunk, chunk
            chunks.append(chunk)
    assert done and chunks[-1]["choices"][0]["finish_reason"] in {"stop", "length"}
    print("STREAM:", "".join(c["choices"][0]["delta"].get("content", "") for c in chunks), "DONE:", done)
r = requests.post(base_url + "/v1/chat/completions", json={**payload, "stop": "4"}, timeout=180)
r.raise_for_status()
assert "4" not in (r.json()["choices"][0]["message"]["content"] or "")
print("STOP: passed")
tool_payload = {**payload, "max_tokens": 128, "messages": [{"role": "user", "content": "Use get_weather to check the weather in Paris. Return the tool call only."}], "tools": [{"type": "function", "function": {"name": "get_weather", "description": "Get current weather in a city", "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}]}
r = requests.post(base_url + "/v1/chat/completions", json=tool_payload, timeout=180)
r.raise_for_status()
print("TOOLS:", r.json()["choices"][0])
assert r.json()["choices"][0]["finish_reason"] == "tool_calls"

resp_payload = {"model": API_MODEL, "input": "What is 2 + 2? Reply with only the number.", "max_output_tokens": 32, "temperature": 0}
r = requests.post(base_url + "/v1/responses", json=resp_payload, timeout=180)
r.raise_for_status()
body = r.json()
assert body["object"] == "response" and body["status"] in {"completed", "incomplete"}
assert body["output"][0]["type"] == "message"
assert body["output"][0]["content"][0]["type"] == "output_text"
print("RESPONSES:", body["output"][0]["content"][0]["text"])
got = requests.get(base_url + "/v1/responses/" + body["id"], timeout=10)
got.raise_for_status()
assert got.json()["id"] == body["id"]
items = requests.get(base_url + "/v1/responses/" + body["id"] + "/input_items", timeout=10)
items.raise_for_status()
assert items.json()["object"] == "list" and items.json()["data"]
follow = requests.post(base_url + "/v1/responses", json={**resp_payload, "input": "Thanks.", "previous_response_id": body["id"]}, timeout=180)
follow.raise_for_status()
assert follow.json()["previous_response_id"] == body["id"]
print("RESPONSES RETRIEVE/FOLLOW-UP: passed")
with requests.post(base_url + "/v1/responses", json={**resp_payload, "stream": True}, stream=True, timeout=180) as r:
    r.raise_for_status()
    events, done = [], False
    for line in r.iter_lines():
        if line == b"data: [DONE]":
            done = True
        elif line.startswith(b"data: "):
            event = json.loads(line[6:])
            assert event.get("type") != "error", event
            events.append(event)
    types = [e.get("type") for e in events]
    assert done and "response.created" in types and "response.completed" in types
    text = "".join(e.get("delta", "") for e in events if e.get("type") == "response.output_text.delta")
    print("RESPONSES STREAM:", text, "DONE:", done)
resp_tools = {"model": API_MODEL, "input": "Use get_weather to check the weather in Paris. Return the tool call only.",
    "max_output_tokens": 128, "temperature": 0,
    "tools": [{"type": "function", "name": "get_weather", "description": "Get current weather in a city",
               "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}]}
r = requests.post(base_url + "/v1/responses", json=resp_tools, timeout=180)
r.raise_for_status()
print("RESPONSES TOOLS:", r.json()["output"])
assert r.json()["output"][0]["type"] == "function_call"
assert r.json()["output"][0]["name"] == "get_weather"
print("LOCAL API CHECKS PASSED")


In [ ]:
# Authenticate ngrok without putting the token in notebook source.
token = load_secret("NGROK_AUTHTOKEN", "NGROK_TOKEN")
if not token:
    raise RuntimeError(
        "Missing NGROK_AUTHTOKEN. In this Kaggle notebook open Add-ons → Secrets and add NGROK_AUTHTOKEN."
    )
from pyngrok import ngrok, conf
conf.get_default().auth_token = token
print("ngrok token configured")


In [ ]:
# Publish only the model API through ngrok (no API key, matching the Colab demo).
assert api_server.started, "Run the API server cell first"
if "tunnel" in globals():
    ngrok.disconnect(tunnel.public_url)
tunnel = ngrok.connect(addr="http://127.0.0.1:8000", proto="http", bind_tls=True)
public_url = tunnel.public_url
print("PUBLIC BASE URL:", public_url + "/v1")
print("MODEL:", API_MODEL)
print("DOCS:", public_url + "/docs")
print("API key for OpenAI clients: not-needed")
print("CHAT COMPLETIONS:", public_url + "/v1/chat/completions")
print("RESPONSES:", public_url + "/v1/responses")
print("Limits: text only; 2048 input tokens; 512 output tokens; one request at a time.")
# Verify a full request through the public tunnel.
check = requests.post(public_url + "/v1/chat/completions",
    headers={"ngrok-skip-browser-warning": "true"},
    json={"model": API_MODEL, "messages": [{"role": "user", "content": "What is 2+2? Reply with only the number."}], "temperature": 0, "max_tokens": 16},
    timeout=180)
check.raise_for_status()
print("PUBLIC TEST:", check.json()["choices"][0]["message"]["content"])
# To stop public access, run: ngrok.disconnect(tunnel.public_url)
# To stop the local API too, run: api_server.should_exit = True
